# Manual Single-File Skim

Select an MC set and flavor - the notebook picks an example file and skims it with `FilterFrame`.  
For testing a single file without submitting a batch job.

In [ ]:
MC       = "Spring2026MC"        # "Spring2026MC"  |  "340StringMC"
FLAVOR   = "Muon"                # "Muon" | "Electron" | "Tau" | "NC"
GEOMETRY = "strings_102_40m"     # sub-geometry key matching paths.py + Metadata/GeometryFiles/<MC>/<GEOMETRY>.csv

TASK_ID  = 0                     # index of the file to pick from the sorted input list

FILTERFRAME_PY = "/project/def-nahee/kbas/Graphnet-Applications/DataPreperation/Skim/FilterFrame.py"

In [2]:
import sys
import re
from pathlib import Path

PROJECT_ROOT = Path("/project/def-nahee/kbas/Graphnet-Applications")

sys.path.insert(0, str(PROJECT_ROOT / "Metadata"))
import paths

# raw MC input (full geometry — always skimmed from here)
if MC == "Spring2026MC":
    full_entry = paths.SPRING2026MC_I3["full_geometry"][FLAVOR]
    geom_entry = paths.SPRING2026MC_I3[GEOMETRY][FLAVOR]
elif MC == "340StringMC":
    full_entry = paths.STRING340MC_I3["full_geometry"][FLAVOR]
    geom_entry = paths.STRING340MC_I3[GEOMETRY][FLAVOR]
else:
    raise ValueError(f"Unknown MC: {MC}")

INDIR   = Path(full_entry["path"])
FMT     = full_entry["format"]     # "zst" or "i3"
PATTERN = f"*.i3.{FMT}" if FMT != "i3" else "*.i3"

GCD     = Path(paths.GCD[MC])

SELECTION_CSV = PROJECT_ROOT / "Metadata" / "GeometryFiles" / MC / f"{GEOMETRY}.csv"

# output directory — same path as the batch pipeline uses for this geometry/flavor
OUTDIR  = Path(geom_entry["path"])

print(f"MC            : {MC}")
print(f"FLAVOR        : {FLAVOR}")
print(f"GEOMETRY      : {GEOMETRY}")
print(f"INDIR         : {INDIR}")
print(f"PATTERN       : {PATTERN}")
print(f"GCD           : {GCD}")
print(f"SELECTION_CSV : {SELECTION_CSV}")
print(f"OUTDIR        : {OUTDIR}")

for label, p in [("INDIR", INDIR), ("GCD", GCD), ("SELECTION_CSV", SELECTION_CSV)]:
    status = "OK" if Path(p).exists() else "MISSING"
    print(f"  [{status}] {label}")

MC            : Spring2026MC
FLAVOR        : Muon
GEOMETRY      : strings_102_40m
INDIR         : /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator
PATTERN       : *.i3.zst
GCD           : /project/6008051/pone_simulation/GCD_Library/PONE_800mGrid_40mSpacing_40OMstring.i3.gz
SELECTION_CSV : /project/def-nahee/kbas/Graphnet-Applications/Metadata/GeometryFiles/Spring2026MC/strings_102_40m.csv
OUTDIR        : /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons
  [OK] INDIR
  [OK] GCD
  [OK] SELECTION_CSV


In [3]:
# list input files and pick one by TASK_ID

all_files = sorted(INDIR.rglob(PATTERN))

print(f"Found {len(all_files)} files matching '{PATTERN}' in {INDIR}")

if not all_files:
    raise FileNotFoundError("No files found — check INDIR and PATTERN")

if not (0 <= TASK_ID < len(all_files)):
    raise IndexError(f"TASK_ID={TASK_ID} out of range (0..{len(all_files)-1})")

INFILE = all_files[TASK_ID]
print(f"\nSelected file (TASK_ID={TASK_ID}):")
print(f"  {INFILE}")

Found 69 files matching '*.i3.zst' in /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator

Selected file (TASK_ID=0):
  /project/6008051/pone_simulation/MC000008-nu_mu-2_6-LeptonInjector_PROPOSAL_clsim-v17.1/Generator/38460126/gen_001.i3.zst


In [4]:
# parse allowed string IDs from selection CSV

csv_text = SELECTION_CSV.read_text()
nums = list(map(int, re.findall(r"\d+", csv_text)))

seen = set()
ALLOWED_STRINGS = []
for n in nums:
    if n not in seen:
        seen.add(n)
        ALLOWED_STRINGS.append(n)

print(f"Allowed strings ({len(ALLOWED_STRINGS)}): {ALLOWED_STRINGS[:10]}{'...' if len(ALLOWED_STRINGS) > 10 else ''}")

Allowed strings (102): [414, 415, 421, 422, 451, 452, 453, 458, 459, 460]...


In [5]:
# import FilterFrame and icetray

filterframe_path = Path(FILTERFRAME_PY).resolve()
if not filterframe_path.exists():
    raise FileNotFoundError(f"FilterFrame.py not found: {filterframe_path}")

sys.path.insert(0, str(filterframe_path.parent))
from FilterFrame import FilterFrame  # type: ignore  # noqa

from icecube import icetray, dataio  # noqa
from icecube.icetray import I3LogLevel

icetray.I3Logger.global_logger.set_level(I3LogLevel.LOG_WARN)

print("FilterFrame imported OK")

FilterFrame imported OK


In [6]:
# resolve output path (same naming convention as trim_I3.py)

OUTDIR.mkdir(parents=True, exist_ok=True)

rel = INFILE.relative_to(INDIR)
parts = list(rel.parts)
suffixes = "".join(INFILE.suffixes)
parts[-1] = parts[-1][:-len(suffixes)] if suffixes else INFILE.stem
stem = "_".join(parts)

OUTFILE = OUTDIR / f"{FLAVOR.lower()}_{stem}.i3.gz"

print(f"Output file: {OUTFILE}")

Output file: /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons/muon_38460126_gen_001.i3.gz


In [7]:
# run skim

import time

print(f"Skimming: {INFILE.name}")
print(f"-> {OUTFILE}")

t0 = time.time()

tray = icetray.I3Tray()
tray.Add("I3Reader", FilenameList=[str(GCD), str(INFILE)])
tray.Add(FilterFrame, AllowedStrings=ALLOWED_STRINGS, OnlyDAQ=True)
tray.Add(
    "I3Writer",
    Filename=str(OUTFILE),
    Streams=[
        icetray.I3Frame.TrayInfo,
        icetray.I3Frame.Simulation,
        icetray.I3Frame.DAQ,
    ],
)
tray.Execute()
tray.Finish()

elapsed = time.time() - t0
size_mb = OUTFILE.stat().st_size / 1e6
print(f"Done in {elapsed:.1f}s — output size: {size_mb:.1f} MB")

Skimming: gen_001.i3.zst
-> /scratch/kbas/Spring2026MC/Strings_102_40m/Muon_I3Photons/muon_38460126_gen_001.i3.gz
Done in 4.5s — output size: 30.9 MB
